<a href="https://colab.research.google.com/github/postnicov/ResazurinResorufin/blob/main/Image_Color_Analysis_v1_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image colour analysis v1.1 — median RGB, CIE Lab (D65), and Munsell (Illuminant C)

Upload one ZIP archive manually. The notebook analyses every supported image within it, computes the per-channel median RGB values over pixels with nonzero alpha, reports CIE Lab relative to D65, and obtains Munsell notation after Bradford/Von Kries adaptation from D65 to Illuminant C. Results are exported to a formatted Excel workbook.

After uploading the ZIP, run the optional ColorChecker cell. Correction is off by default. To enable it, check the box and upload the transform JSON exported by the ColorChecker calibration notebook. Apply only to images acquired under matching lighting, exposure, camera, and rendering settings. With correction enabled, embedded profiles are converted to sRGB; untagged non-CMYK images are assumed sRGB. Alpha is used only to exclude fully transparent pixels. The transform is applied to each visible pixel before taking medians; the existing Lab(D65), Munsell(Illuminant C), and Excel swatches then use those medians. No source images are overwritten.

In [1]:
#@title 1. Install and import required packages
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    'PIL': 'Pillow',
    'colour': 'colour-science',
    'pandas': 'pandas',
    'openpyxl': 'openpyxl',
}
missing_packages = [
    package for module, package in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing_packages])

from datetime import datetime
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile
import re
import warnings

import colour
from colour.adaptation import chromatic_adaptation_VonKries
from colour.utilities import ColourUsageWarning
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from google.colab import files

warnings.filterwarnings('ignore', category=ColourUsageWarning)
print('Dependencies are ready.')

Dependencies are ready.


In [2]:
#@title 2. Set analysis options
REPORT_TITLE = 'Median Image Colour Analysis'
SUPPORTED_IMAGE_EXTENSIONS = {'.bmp', '.gif', '.jpeg', '.jpg', '.png', '.tif', '.tiff', '.webp'}
OBSERVER = 'CIE 1931 2 Degree Standard Observer'
CAT_TRANSFORM = 'Bradford'

# RGB and CIE Lab are referenced to D65. Munsell renotation data are defined under C.
D65_XY = colour.CCS_ILLUMINANTS[OBSERVER]['D65']
C_XY = colour.CCS_ILLUMINANTS[OBSERVER]['C']
D65_WHITE_XYZ = colour.xy_to_XYZ(D65_XY)
C_WHITE_XYZ = colour.xy_to_XYZ(C_XY)

print('RGB / CIE Lab illuminant: D65')
print('Munsell illuminant: C')
print('Chromatic adaptation: Von Kries with Bradford transform')
print('Supported formats:', ', '.join(sorted(SUPPORTED_IMAGE_EXTENSIONS)))

RGB / CIE Lab illuminant: D65
Munsell illuminant: C
Chromatic adaptation: Von Kries with Bradford transform
Supported formats: .bmp, .gif, .jpeg, .jpg, .png, .tif, .tiff, .webp


In [5]:
#@title 3. Define image-analysis, colour-conversion, and Excel-export functions
HUE_SECTORS = ['R', 'YR', 'Y', 'GY', 'G', 'BG', 'B', 'PB', 'P', 'RP']

def safe_extract_zip(zip_path, destination):
    destination = Path(destination).resolve()
    with ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f'Unsafe ZIP member path: {member.filename}')
        archive.extractall(destination)

def median_rgb(image_path, transform=None):
    """Return median over pixels with nonzero alpha 8-bit RGB values from the first image frame."""
    with Image.open(image_path) as image:
        image = ImageOps.exif_transpose(image)
        if transform is None:
            rgba = np.asarray(image.convert('RGBA'), dtype=np.uint8).reshape(-1, 4)
            visible = rgba[rgba[:, 3] > 0, :3]
        else:
            visible = visible_srgb_pixels(image)
    if transform is not None:
        for start in range(0, len(visible), 262144):
            chunk = visible[start:start+262144]
            chunk[:] = colorchecker_rgb8(chunk, transform)
    if visible.size == 0:
        raise ValueError('The image has no non-transparent pixels.')
    return tuple(np.rint(np.median(visible, axis=0)).astype(int))

def format_munsell_number(number):
    number = float(number)
    if np.isclose(number, round(number)):
        return str(int(round(number)))
    return f'{number:.2f}'.rstrip('0').rstrip('.')

def normalise_zero_hue_to_previous_sector(munsell_notation, tolerance=1e-8):
    text = str(munsell_notation).strip().upper()
    match = re.fullmatch(
        r'([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*([A-Z]+)\s+([^\s/]+)\s*/\s*([^\s]+)',
        text,
    )
    if match is None:
        return text
    hue_number_text, sector, value_text, chroma_text = match.groups()
    hue_number = float(hue_number_text)
    if np.isclose(hue_number, 0.0, atol=tolerance) and sector in HUE_SECTORS:
        prior_sector = HUE_SECTORS[(HUE_SECTORS.index(sector) - 1) % len(HUE_SECTORS)]
        return f'10{prior_sector} {value_text}/{chroma_text}'
    return f'{format_munsell_number(hue_number)}{sector} {value_text}/{chroma_text}'

def round_to_half(value):
    return np.floor(float(value) * 2.0 + 0.5) / 2.0

def format_half(value):
    value = round_to_half(value)
    return str(int(round(value))) if np.isclose(value, round(value)) else f'{value:.1f}'

def munsell_to_half_step(code):
    neutral = re.fullmatch(r'N\s*([0-9.]+)', str(code).strip().upper())
    if neutral:
        return f'N {format_half(neutral.group(1))}'
    chromatic = re.fullmatch(r'([0-9.]+)\s*([A-Z]+)\s+([0-9.]+)\s*/\s*([0-9.]+)', str(code).strip().upper())
    if chromatic is None:
        return str(code)
    hue, sector, value, chroma = chromatic.groups()
    return f'{format_half(hue)}{sector} {format_half(value)}/{format_half(chroma)}'

def rgb_to_lab_and_munsell(rgb):
    """Convert median sRGB to Lab(D65), then Munsell(C) via Bradford/Von Kries CAT."""
    rgb_normalised = np.asarray(rgb, dtype=float) / 255.0
    xyz_d65 = colour.sRGB_to_XYZ(rgb_normalised)
    lab_d65 = colour.XYZ_to_Lab(xyz_d65, illuminant=D65_XY)

    if np.max(np.abs(lab_d65[1:])) < 1.5:
        return tuple(np.round(lab_d65, 2)), f'N {lab_d65[0]:.1f}', ''

    # The Munsell conversion must receive XYZ referenced to Illuminant C.
    xyz_c = chromatic_adaptation_VonKries(
        xyz_d65, D65_WHITE_XYZ, C_WHITE_XYZ, transform=CAT_TRANSFORM
    )
    xyY_c = colour.XYZ_to_xyY(xyz_c)
    try:
        raw_code = colour.xyY_to_munsell_colour(xyY_c)
        return tuple(np.round(lab_d65, 2)), normalise_zero_hue_to_previous_sector(raw_code), ''
    except Exception:
        # Use only a minimally altered C-referenced coordinate for an approximate Munsell code.
        lab_for_search = np.array([np.clip(lab_d65[0], 10.0, 90.0), lab_d65[1], lab_d65[2]])
        lightness_clipped = not np.isclose(lab_for_search[0], lab_d65[0])
        for chroma_scale in np.linspace(1.0, 0.0, 101):
            candidate_lab = np.array([lab_for_search[0], lab_for_search[1] * chroma_scale, lab_for_search[2] * chroma_scale])
            candidate_xyz_d65 = colour.Lab_to_XYZ(candidate_lab, illuminant=D65_XY)
            candidate_xyz_c = chromatic_adaptation_VonKries(
                candidate_xyz_d65, D65_WHITE_XYZ, C_WHITE_XYZ, transform=CAT_TRANSFORM
            )
            try:
                code = colour.xyY_to_munsell_colour(colour.XYZ_to_xyY(candidate_xyz_c))
                comment = 'Approximate Munsell code: nearest valid renotation coordinate used'
                if lightness_clipped:
                    comment += '; lightness was constrained to the Munsell conversion range'
                return tuple(np.round(lab_d65, 2)), normalise_zero_hue_to_previous_sector(code), comment
            except Exception:
                continue
        return tuple(np.round(lab_d65, 2)), 'Unavailable', 'No valid Munsell renotation coordinate could be found'

def roman_to_integer(roman):
    values = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}
    total, previous = 0, 0
    for character in reversed(roman):
        value = values[character]
        total += -value if value < previous else value
        previous = max(previous, value)
    return total

def identifier_sort_key(row):
    identifier = row['ID']
    match = re.fullmatch(r'([IVXLCDM]+)([a-z]?)', identifier, flags=re.IGNORECASE)
    if match:
        roman, suffix = match.groups()
        return (0, suffix.casefold(), roman_to_integer(roman.upper()), identifier.casefold())
    return (1, identifier.casefold(), 0, identifier.casefold())

def build_excel(rows, output_path, title):
    columns = ['ID', 'R', 'G', 'B', 'L*', 'a*', 'b*', 'Munsell code', 'Munsell code (0.5)', 'Munsell comment', 'Median RGB colour']
    dataframe = pd.DataFrame(rows, columns=columns)
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        dataframe.to_excel(writer, sheet_name='Colour analysis', index=False, startrow=2)
        worksheet = writer.book['Colour analysis']
        worksheet['A1'] = title
        worksheet['A2'] = 'Median RGB channels; CIE Lab(D65); Munsell(C) obtained by Bradford/Von Kries adaptation from D65 to C.'
        worksheet.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(columns))
        worksheet.merge_cells(start_row=2, start_column=1, end_row=2, end_column=len(columns))
        worksheet['A1'].font = Font(bold=True, size=14)
        worksheet['A2'].alignment = Alignment(wrap_text=True)
        header_row = 3
        for cell in worksheet[header_row]:
            cell.font = Font(bold=True)
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        colour_column = columns.index('Median RGB colour') + 1
        for row_index, row in enumerate(rows, start=header_row + 1):
            colour_cell = worksheet.cell(row=row_index, column=colour_column)
            r, g, b = row['R'], row['G'], row['B']
            colour_cell.fill = PatternFill(fill_type='solid', fgColor=f'{r:02X}{g:02X}{b:02X}')
            colour_cell.alignment = Alignment(horizontal='center', vertical='center')
        widths = [14, 9, 9, 9, 10, 10, 10, 20, 22, 58, 22]
        for index, width in enumerate(widths, start=1):
            worksheet.column_dimensions[get_column_letter(index)].width = width
        worksheet.freeze_panes = 'A4'
        worksheet.auto_filter.ref = f'A3:{get_column_letter(len(columns))}{header_row + len(rows)}'
        for row in worksheet.iter_rows(min_row=4, max_row=header_row + len(rows), min_col=1, max_col=len(columns)):
            for cell in row:
                cell.alignment = Alignment(vertical='center', wrap_text=True)

def analyse_zip(zip_path, source_name, transform=None):
    with TemporaryDirectory() as temporary_directory:
        extraction_directory = Path(temporary_directory) / 'images'
        extraction_directory.mkdir()
        safe_extract_zip(zip_path, extraction_directory)
        image_paths = sorted(
            path for path in extraction_directory.rglob('*')
            if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS
        )
        if not image_paths:
            raise ValueError('No supported image files were found in the ZIP archive.')
        rows, failed_files = [], []
        for image_path in image_paths:
            try:
                rgb = median_rgb(image_path, transform)
                lab, munsell_code, comment = rgb_to_lab_and_munsell(rgb)
                rows.append({
                    'ID': image_path.stem, 'R': rgb[0], 'G': rgb[1], 'B': rgb[2],
                    'L*': lab[0], 'a*': lab[1], 'b*': lab[2],
                    'Munsell code': munsell_code,
                    'Munsell code (0.5)': munsell_to_half_step(munsell_code),
                    'Munsell comment': comment,
                    'Median RGB colour': '',
                })
            except Exception as error:
                failed_files.append(f'{image_path.name}: {error}')
        if not rows:
            raise RuntimeError('No images could be analysed. ' + ' | '.join(failed_files))
        rows.sort(key=identifier_sort_key)
        safe_name = re.sub(r'[^A-Za-z0-9_-]+', '_', Path(source_name).stem) or 'images'
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_path = Path('/content') / f'{safe_name}_median_colour_analysis_{timestamp}.xlsx'
        build_excel(rows, output_path, f'{REPORT_TITLE}: {Path(source_name).stem}')
        if transform is not None:
            workbook = load_workbook(output_path)
            metadata = workbook.create_sheet('ColorChecker correction')
            for label, value in [('Notebook version', '1.1'), ('Correction', 'Enabled'),
                                 ('Transform JSON', COLORCHECKER_JSON_NAME),
                                 ('JSON SHA-256', COLORCHECKER_JSON_SHA256), ('Model', transform[0]),
                                 ('Sampling', 'Correct each visible pixel before taking channel medians')]:
                metadata.append([label, None])
                cell = metadata.cell(metadata.max_row, 2)
                cell.value = value
                cell.data_type = 's'
            metadata.column_dimensions['A'].width = 24
            metadata.column_dimensions['B'].width = 70
            workbook.save(output_path)
        preview = pd.DataFrame(rows).drop(columns=['Median RGB colour'])
        return output_path, preview, failed_files

In [9]:
#@title 4. Upload the ZIP archive of images
zip_ready = False
COLORCHECKER_TRANSFORM = None
COLORCHECKER_JSON_NAME = None
COLORCHECKER_JSON_SHA256 = None
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(uploaded) != 1 or len(zip_names) != 1:
    raise ValueError('Upload exactly one ZIP file containing the images to analyse.')

zip_name = zip_names[0]
zip_path = Path('/content') / zip_name
zip_path.write_bytes(uploaded[zip_name])
zip_ready = True
print(f'Image archive ready: {zip_name}')


Saving Kfr.zip to Kfr (1).zip
Image archive ready: Kfr (1).zip


In [10]:
#@title 5. Optional ColorChecker transform — enable and upload JSON
import io
import json
import hashlib
from PIL import ImageCms

APPLY_COLORCHECKER = False #@param {type:"boolean"}
COLORCHECKER_TRANSFORM = None
COLORCHECKER_JSON_NAME = None
COLORCHECKER_JSON_SHA256 = None

FEATURE_ORDERS = {
    'matrix': ['R', 'G', 'B'],
    'affine': ['R', 'G', 'B', '1'],
    'root_polynomial': ['R', 'G', 'B', 'sqrt(RG)', 'sqrt(RB)', 'sqrt(GB)', '1'],
}

def parse_colorchecker_json(raw):
    t = json.loads(raw)
    if not isinstance(t, dict) or t.get('schema') not in (
        'colorchecker_srgb_transform/1.0', 'colorchecker_srgb_transform/1.1'
    ):
        raise ValueError('Expected a ColorChecker transform JSON with schema 1.0 or 1.1.')
    model = t.get('model')
    if not isinstance(model, str) or model not in FEATURE_ORDERS or t.get('feature_order') != FEATURE_ORDERS[model]:
        raise ValueError('Unsupported model or incorrect feature order in JSON.')
    coef = np.asarray(t.get('coefficient_matrix'), dtype=float)
    if coef.shape != (len(FEATURE_ORDERS[model]), 3) or not np.isfinite(coef).all():
        raise ValueError('Coefficient matrix has the wrong dimensions or non-finite entries.')
    return model, coef

def colorchecker_rgb8(rgb, transform):
    model, coef = transform
    s = np.asarray(rgb, dtype=np.float64) / 255.0
    x = np.where(s <= 0.04045, s/12.92, ((s+0.055)/1.055)**2.4)
    if model == 'matrix':
        features = x
    elif model == 'affine':
        features = np.column_stack((x, np.ones(len(x))))
    else:
        r, g, b = x.T
        features = np.column_stack((r, g, b, np.sqrt(r*g), np.sqrt(r*b), np.sqrt(g*b), np.ones(len(x))))
    with np.errstate(over='raise', invalid='raise'):
        y = np.clip(features @ coef, 0, 1)
    encoded = np.where(y <= 0.0031308, 12.92*y, 1.055*y**(1/2.4)-0.055)
    return np.rint(255*encoded).astype(np.uint8)

def visible_srgb_pixels(image):
    if image.mode not in ('RGB', 'RGBA', 'L', 'LA', 'P', 'CMYK', '1', 'PA'):
        raise ValueError('ColorChecker correction requires an 8-bit rendered image.')
    icc = image.info.get('icc_profile')
    rgba = image.convert('RGBA')
    keep = np.asarray(rgba.getchannel('A')).reshape(-1) > 0
    if image.mode == 'CMYK' and not icc:
        raise ValueError('ColorChecker correction requires an ICC profile for CMYK.')
    plane = image if image.mode in ('CMYK', 'RGB', 'L') else (
        image.convert('L') if image.mode in ('LA', '1') else rgba.convert('RGB'))
    if icc:
        plane = ImageCms.profileToProfile(plane, ImageCms.ImageCmsProfile(io.BytesIO(icc)),
                                          ImageCms.createProfile('sRGB'), outputMode='RGB')
    else:
        plane = plane.convert('RGB')
    return np.asarray(plane, dtype=np.uint8).reshape(-1, 3)[keep]

if not globals().get('zip_ready', False):
    raise RuntimeError('Upload the image ZIP in the preceding cell first.')
if APPLY_COLORCHECKER:
    json_files = files.upload()
    if len(json_files) != 1:
        raise ValueError('Upload exactly one ColorChecker transform JSON.')
    name, raw = next(iter(json_files.items()))
    if Path(name).suffix.lower() != '.json':
        raise ValueError('Expected a .json file.')
    COLORCHECKER_TRANSFORM = parse_colorchecker_json(raw)
    COLORCHECKER_JSON_NAME = Path(name).name
    COLORCHECKER_JSON_SHA256 = hashlib.sha256(raw).hexdigest()
    print(f'Correction enabled: {COLORCHECKER_JSON_NAME} ({COLORCHECKER_TRANSFORM[0]}).')
else:
    print('Correction disabled; original analysis is unchanged.')


Correction disabled; original analysis is unchanged.


In [11]:
#@title 6. Analyse images and download the Excel report
if not globals().get("zip_ready", False):
    raise RuntimeError("Upload the ZIP first.")
if globals().get("APPLY_COLORCHECKER", False) and COLORCHECKER_TRANSFORM is None:
    raise RuntimeError("Run the optional correction cell and upload a valid JSON, or turn correction off.")
output_path, preview, failed_files = analyse_zip(zip_path, zip_name, COLORCHECKER_TRANSFORM)

if failed_files:
    print('Files skipped because they could not be read:')
    print('\n'.join(failed_files))
display(preview)
print(f'Excel report created: {output_path.name}')
files.download(str(output_path))

,ID,R,G,B,L*,a*,b*,Munsell code,Munsell code (0.5),Munsell comment
0,Kfr,118,101,130,45.32,12.66,-13.59,4.4P 4.5/4.0,4.5P 4.5/4,


Excel report created: Kfr_1__median_colour_analysis_20261118_174742.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Notes

- No online archive is retrieved: input is exclusively a ZIP file uploaded through the ZIP upload cell.
- The reported RGB values are per-channel medians of non-transparent image pixels.
- sRGB-to-XYZ and CIE Lab calculations are referenced to D65. For Munsell conversion, XYZ(D65) is adapted to XYZ(C) with Von Kries chromatic adaptation using the Bradford transform, before invoking the Munsell renotation conversion.
- If direct Munsell conversion is outside the supported renotation range, the workbook leaves the code in its own cell and places the approximation explanation in the separate **Munsell comment** cell.
- The Excel colour column is filled with the corresponding median RGB colour rather than embedding a rectangle image.